# 05 - M5: plain Mamba block (real shapes, real T4)
**Runtime > T4 GPU.** Gate: scan is causal (perturbing token t changes nothing before t); matches an independent reference scan on random input; memory OK.
'Memory OK' (D008) = full pipeline CLIP+LoRA -> front end -> 2 Mamba blocks, batch 8, fp16 autocast, forward+backward peak **< 10 GiB** (T4 has ~15 GiB; M4 alone peaked at 5.63 GiB).
Also records speed for chunk sizes and the gradient-checkpoint option so M8 can pick a training config.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import json, time, torch
from src.mamba import MambaBlock, PlainMamba, selective_scan
from src.mamba_ref import scan_reference
dev = 'cuda'
torch.manual_seed(0)
res = {'milestone': 'M5'}

In [ ]:
# GATE 1: matches the independent float64 sequential reference at REAL size (L=576, d_inner=768, N=16), on GPU data
blk = MambaBlock(384).to(dev).eval()
b, L, di, N = 2, 576, blk.d_inner, blk.d_state
xc = torch.randn(b, L, di, device=dev); dt = torch.rand(b, L, di, device=dev) * 0.1 + 1e-3
Bm, Cm = torch.randn(b, L, N, device=dev), torch.randn(b, L, N, device=dev)
A = -torch.exp(blk.A_log)
with torch.no_grad():
    ref = scan_reference(xc, dt, A, Bm, Cm, blk.D)
    errs = {}
    for ck in (8, 16):
        y, _ = selective_scan(xc, dt, A, Bm, Cm, blk.D, chunk=ck)
        errs[ck] = (y.double().cpu() - ref).abs().max().item()
print('max abs err vs reference', errs, '| scale', ref.abs().max().item())
assert all(e < 1e-3 for e in errs.values())
res['ref_max_abs_err'] = errs
print('GATE 1 OK')

In [ ]:
# GATE 2: causality at real size, fp32 and fp16 autocast, perturbing tokens at chunk edges and the very end
m = PlainMamba(2).to(dev).eval()
x = torch.randn(2, 576, 384, device=dev)
worst = {}
for mode in ('fp32', 'fp16'):
    for t in (0, 1, 15, 16, 287, 575):
        x2 = x.clone(); x2[:, t] += 5.0
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=(mode == 'fp16')):
            a, b2 = m(x), m(x2)
        before = (a[:, :t].float() - b2[:, :t].float()).abs().max().item() if t else 0.0
        after = (a[:, t:].float() - b2[:, t:].float()).abs().max().item()
        worst[f'{mode}@{t}'] = (before, after)
        assert before < 1e-4, (mode, t, before)   # nothing before t moves
        assert after > 1e-3, (mode, t, after)     # but t and later do (block is alive)
print({k: (f'{v[0]:.2e}', f'{v[1]:.2e}') for k, v in worst.items()})
res['causality_before_after'] = worst
print('GATE 2 OK')

In [ ]:
# GATE 3: memory + speed. Real pipeline: CLIP+LoRA -> front end (M4) -> 2 Mamba blocks, B=8, fp16 autocast, fwd+bwd
import numpy as np
from datasets import load_dataset
from transformers import CLIPVisionModel
from src.data import load_manifest, ManifestDataset
from src.backbone import ClipLoRABackbone
from src import concepts as K
from src.fusion import ConceptFrontEnd
import os
from transformers import CLIPTextModelWithProjection, CLIPTokenizer
ASSET = 'assets/concepts_text_768.npy'          # committed after M4; if absent, encode it now (same code as notebook 04)
if os.path.exists(ASSET):
    T = K.load_or_encode(ASSET)
else:
    tm = CLIPTextModelWithProjection.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32)
    T = K.load_or_encode(ASSET, tm, CLIPTokenizer.from_pretrained(K.TEXT_CHECKPOINT), device='cuda'); del tm
mf = load_manifest('Midjourney')
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=mf['revision'])
tr = ManifestDataset(mf, 'train', ds['train'], ds['validation'], protocol='controlled', train=False)
xb = torch.stack([tr[i][0] for i in range(8)]).to(dev)
clip = CLIPVisionModel.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32).to(dev).eval()
bb = ClipLoRABackbone(clip, taps=(16, 24), r=8, alpha=16.0).to(dev)
fe = ConceptFrontEnd(T, d_in=1024, d=384, n_taps=2).to(dev)

def run(chunk, ckpt, steps=3):
    mm = PlainMamba(2, chunk=chunk, checkpoint=ckpt).to(dev)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    ts = []
    for _ in range(steps):
        for mod in (bb, fe, mm): mod.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast('cuda', dtype=torch.float16):
            H = mm(fe(bb(xb))['X'])
        H.float().square().mean().backward()
        torch.cuda.synchronize(); ts.append(time.time() - t0)
    assert torch.isfinite(H).all()
    return {'peak_gib': torch.cuda.max_memory_allocated() / 2**30, 'sec_per_step_b8': float(np.median(ts)),
            'img_per_s': 8 / float(np.median(ts))}

grid = {}
for chunk, ckpt in ((16, False), (8, False), (16, True)):
    grid[f'chunk{chunk}_ckpt{ckpt}'] = run(chunk, ckpt); print(chunk, ckpt, grid[f'chunk{chunk}_ckpt{ckpt}'])
res['pipeline_b8'] = grid
best = grid['chunk16_ckptFalse']
assert best['peak_gib'] < 10.0, 'default config over budget: switch M8 to the checkpoint=True row'
print('GATE 3 OK | default (chunk 16, no checkpoint) peak', f"{best['peak_gib']:.2f} GiB")

In [ ]:
# Block-only microbenchmark (fwd+bwd, B=8, L=576, d=384, fp16 autocast) vs. a 576-step python loop, for the record
blk = MambaBlock(384).to(dev); xx = torch.randn(8, 576, 384, device=dev, requires_grad=True)
def bench(f, n=3):
    ts = []
    for _ in range(n):
        torch.cuda.synchronize(); t0 = time.time(); f(); torch.cuda.synchronize(); ts.append(time.time() - t0)
    return float(np.median(ts))
def step_chunked():
    with torch.autocast('cuda', dtype=torch.float16): blk(xx).float().sum().backward()
blk_seq = MambaBlock(384, chunk=1).to(dev)   # chunk=1 == the 576-iteration python loop
def step_seq():
    with torch.autocast('cuda', dtype=torch.float16): blk_seq(xx).float().sum().backward()
res['block_bench_sec'] = {'chunk16': bench(step_chunked), 'chunk1_loop': bench(step_seq)}
print(res['block_bench_sec'])

In [ ]:
res['gate'] = 'PASS'
for p in (f'{DRIVE_ROOT}/runs/m5_gate.json', '/content/csm-clip/runs/m5_gate.json'):
    json.dump(res, open(p, 'w'), indent=1)
print(json.dumps(res, indent=1))
# Then: commit runs/m5_gate.json (and the updated docs) from your machine, tag m5-done.